In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import nltk
import spacy

print("HCL project environment is ready!")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

HCL project environment is ready!
Pandas: 3.0.6
NumPy: 2.5.3
Scikit-learn: 1.9.1


In [2]:
from pathlib import Path

print("Project path:", Path.cwd())

Project path: C:\Users\mohammad rayyan\SAFETY_INCIDENT_NLP


In [3]:
from pathlib import Path

gitignore = """
__pycache__/
.ipynb_checkpoints/
.venv/
venv/
.env

# Keep datasets private
DATA/raw/
DATA/processed/

# Ignore trained model files
MODELS/*.joblib
MODELS/*.pkl
MODELS/*.pickle

# Temporary files
.DS_Store
Thumbs.db
"""

Path(".gitignore").write_text(
    gitignore.strip() + "\n",
    encoding="utf-8"
)

print(".gitignore created successfully!")

.gitignore created successfully!


In [1]:
from pathlib import Path

# Locate the raw dataset
raw_dir = Path("DATA") / "raw"

# Find all text reports
files = sorted(
    raw_dir.glob("*.txt"),
    key=lambda p: int(p.stem) if p.stem.isdigit() else p.stem
)

print("Dataset folder:", raw_dir.resolve())
print("Total text reports:", len(files))

# Inspect the first 3 reports
for file in files[:3]:
    text = file.read_text(encoding="utf-8", errors="replace")

    print("\n" + "=" * 60)
    print("File:", file.name)
    print("Characters:", len(text))
    print("Preview:")
    print(text[:500])

Dataset folder: C:\Users\mohammad rayyan\SAFETY_INCIDENT_NLP\DATA\raw
Total text reports: 112

File: 1.txt
Characters: 495
Preview:
德齐龙化工集团一厂三车间在工作压力正常的情况下(该系统设计压力为32MPa，实际操作压力为28.6-29.2MPa)，2号4M40压缩机六段油分出口止回阀法兰与进口管（管道直径102mm）突然脱落，造成氮氢气体（氮气25%、氢气73%、甲烷2%）瞬间泄漏，大量泄漏的氢气与空气混合达到爆炸极限（4—74.2%），遇被弹出的止回阀摩擦撞击铁管产生火花引发爆炸事故，造成2人死亡，11人受伤。因爆炸所形成的冲击波，造成车间后面的煤气管道震裂泄漏引发大火。事故发生后，德齐龙化工集团迅速启动厂内事故应急救援预案，厂内消防员、消防车及时赶赴现场灭火的同时，及时报警增援。县委、县政府及有关部门主要领导闻讯后，及时赶往现场，立即启动县级应急救援预案，冒着生命危险指挥抢险救护工作，受伤职工被立即送往医院，得到了及时抢救和治疗。在消防官兵和本厂职工的共同努力下，10时40分，明火全部扑灭，消除了安全隐患，抢险救援工作结束。法兰内螺纹小径偏差超标较大是导致进口管与法兰脱落的主要原因（法兰内螺纹0°～180°超差1.538mm、90°～270°超差1.618mm），从而也是导致爆炸事故发生的主要原因。

File: 2.txt
Characters: 944
Preview:
贵州兴化化工有限责任公司因进行甲醇罐惰性气体保护设施建设，委托湖北省宜都市昌业锅炉设备安装有限公司进行储罐的二氧化碳管道安装工作（据调查该施工单位施工资质已过期）。2008年7月30日，该安装公司在处于生产状况下的甲醇罐区违规将精甲醇c储罐顶部备用短接打开，与二氧化碳管道进行连接配管，管道另一端则延伸至罐外下部，造成罐体内部通过管道与大气直接连通，致使空气进入罐内，与甲醇蒸汽形成爆炸性混合气体。8月2日上午，因气温较高，罐内爆炸性混合气体通过配管外泄，使罐内、管道及管口区域充斥爆炸性混合气体，由于精甲醇c罐旁边又在违规进行电焊等动火作业（据初步调查，动火作业未办理动火证），引起管口区域爆炸性混合气体燃烧，并通过连通管道引发罐内爆炸性混合气体爆炸，罐底部被冲开，

In [2]:
from pathlib import Path
from collections import Counter

raw_dir = Path("DATA") / "raw"

all_files = [p for p in raw_dir.iterdir() if p.is_file()]

print("Total files in raw folder:", len(all_files))
print("\nFile extensions:")
print(Counter(p.suffix.lower() for p in all_files))

print("\nFirst 20 filenames:")
for file in sorted(all_files, key=lambda p: p.name)[:20]:
    print(file.name)

Total files in raw folder: 112

File extensions:
Counter({'.txt': 112})

First 20 filenames:
1.txt
10.txt
100.txt
101.txt
102.txt
103.txt
104.txt
105.txt
106.txt
107.txt
108.txt
109.txt
11.txt
110.txt
111.txt
112.txt
12.txt
13.txt
14.txt
15.txt


In [5]:
import pandas as pd
from pathlib import Path

raw_dir = Path("DATA") / "raw"

files = sorted(
    raw_dir.glob("*.txt"),
    key=lambda p: int(p.stem)
)

reports = []

for file in files:
    text = file.read_text(
        encoding="utf-8",
        errors="replace"
    )

    reports.append({
        "file_name": file.name,
        "text": text,
        "characters": len(text),
        "words": len(text.split())
    })

df = pd.DataFrame(reports)

print("Total reports:", len(df))
print("\nReport length statistics:")
print(df[["characters", "words"]].describe().round(2))

print("\nFirst 5 reports:")
display(df[["file_name", "characters", "words"]].head())

Total reports: 112

Report length statistics:
       characters   words
count      112.00  112.00
mean       902.66    7.62
std        669.47    8.15
min         73.00    1.00
25%        467.25    2.00
50%        650.00    5.00
75%       1131.50   10.00
max       3391.00   46.00

First 5 reports:


,file_name,characters,words
0,1.txt,495,1
1,2.txt,944,1
2,3.txt,1485,24
3,4.txt,1260,1
4,5.txt,913,1


In [7]:
from pathlib import Path
import pandas as pd

file_path = (
    Path.cwd()
    / "DATA"
    / "raw"
    / "accident_hazardous_liquid_jan2010_present.txt"
)

df = pd.read_csv(
    file_path,
    sep="\t",
    encoding="cp1252",
    low_memory=False
)

print("Dataset loaded successfully!")
print("Rows and columns:", df.shape)

display(df.head())

Dataset loaded successfully!
Rows and columns: (6009, 648)


,REPORT_RECEIVED_DATE,IYEAR,REPORT_NUMBER,SUPPLEMENTAL_NUMBER,REPORT_TYPE,OPERATOR_ID,NAME,OPERATOR_STREET_ADDRESS,OPERATOR_CITY_NAME,OPERATOR_STATE_ABBREVIATION,...,PREPARER_FAX,PREPARED_DATE,LOCAL_CONTACT_NAME,LOCAL_CONTACT_EMAIL,LOCAL_CONTACT_TELEPHONE,AUTHORIZER_NAME,AUTHORIZER_TELEPHONE,AUTHORIZER_TITLE,AUTHORIZER_EMAIL,NARRATIVE
0,8/27/2026,2026,20260196,42679,ORIGINAL,30020,W & T OFFSHORE INC,5718 WESTHEIMER ROAD SUITE 700,HOUSTON,TX,...,NaN,8/27/2026,KIRK BORDELON,kbordelon@wtoffshore.com,832-486-3697,JOHN POOLE,713-624-7338,VICE PRESIDENT HSER,jpoole@wtoffshore.com,"DURING ROUTINE INSPECTION OF THE PLATFORM, PLA..."
1,8/30/2026,2026,20260200,42687,ORIGINAL,32109,"ONEOK NGL PIPELINE, LLC",100 WEST FIFTH STREET,TULSA,OK,...,NaN,8/28/2026,NaN,NaN,NaN,RYAN MCCOY,5807410185,NaN,ryan.mccoy@oneok.com,"ON 8/1/26, ONEOK PCC WAS NOTIFIED BY A LANDOWN..."
2,8/28/2026,2026,20260199,42684,ORIGINAL,31957,KINDER MORGAN WINK PIPELINE LLC,"1001 LOUISIANA ST, SUITE 1000",HOUSTON,TX,...,NaN,8/28/2026,NaN,NaN,NaN,DAVID TOWNSEND,7134204491,DIRECTOR - OPERATIONS,david_townsend@kindermorgan.com,"ON JULY 30, 2026, AT 19:10CST, A KINDER MORGAN..."
3,8/13/2026,2026,20260188,42624,ORIGINAL FINAL,39401,GREAT SALT PLAINS PIPELINE LLC,"8301 E 21ST ST, SUITE 370",WICHITA,KS,...,NaN,8/13/2026,JOE VAVERKA,jvaverka@mvpurchasing.com,405-508-0410,ADAM COWART,6208860082,MANAGER OF EHS AND COMPLIANCE,acowart@mvpurchasing.com,"ON JULY 29, 2026, THE CONTROL ROOM RECEIVED A ..."
4,8/27/2026,2026,20260197,42682,ORIGINAL,39377,"WHITNEY OIL & GAS, LLC",1415 LOUSIANA ST,HOUSTON,TX,...,NaN,8/27/2026,TAYLOR WALKER,twalker@torrentoil.com,601-498-7289,RUSSELL STEINER,3375190843,REGULATORY AND COMPLIANCE MANAGER,rsteiner@torrentoil.com,"ON JULY 29, 2026, OPERATING PERSONNEL NOTICED ..."


In [8]:
print("Dataset dimensions:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

Dataset dimensions: (6009, 648)

Column names:
['REPORT_RECEIVED_DATE', 'IYEAR', 'REPORT_NUMBER', 'SUPPLEMENTAL_NUMBER', 'REPORT_TYPE', 'OPERATOR_ID', 'NAME', 'OPERATOR_STREET_ADDRESS', 'OPERATOR_CITY_NAME', 'OPERATOR_STATE_ABBREVIATION', 'OPERATOR_POSTAL_CODE', 'LOCAL_DATETIME', 'TIME_ZONE', 'DAYLIGHT_SAVINGS_IND', 'LOCATION_LATITUDE', 'LOCATION_LONGITUDE', 'COMMODITY_RELEASED_TYPE', 'COMMODITY_SUBTYPE', 'COMMODITY_DETAILS', 'BLEND_DETAILS', 'BIO_DIESEL_DETAILS', 'UNINTENTIONAL_RELEASE_BBLS', 'INTENTIONAL_RELEASE_BBLS', 'RECOVERED_BBLS', 'FATALITY_IND', 'NUM_EMP_FATALITIES', 'NUM_CONTR_FATALITIES', 'NUM_ER_FATALITIES', 'NUM_WORKER_FATALITIES', 'NUM_GP_FATALITIES', 'FATAL', 'INJURY_IND', 'NUM_EMP_INJURIES', 'NUM_CONTR_INJURIES', 'NUM_ER_INJURIES', 'NUM_WORKER_INJURIES', 'NUM_GP_INJURIES', 'INJURE', 'ACCIDENT_IDENTIFIER', 'ACCIDENT_DETAILS', 'OPERATOR_TYPE', 'INCIDENT_IDENTIFIED_DATETIME', 'SYSTEM_PART_INVOLVED', 'ON_OFF_SHORE', 'STATUS_WHEN_IDENTIFIED', 'SHUTDOWN_DUE_ACCIDENT_IND', 'SH

In [9]:
keywords = [
    "ACCIDENT_DETAILS",
    "CAUSE",
    "FAILURE",
    "INJUR",
    "FATAL",
    "RELEASE",
    "ACCIDENT_TYPE",
    "REPORT_TYPE",
    "DATE"
]

for keyword in keywords:
    matching = [
        col for col in df.columns
        if keyword in col.upper()
    ]
    print(f"\n{keyword}:")
    print(matching)


ACCIDENT_DETAILS:
['ACCIDENT_DETAILS']

CAUSE:
['CAUSE', 'CAUSE_DETAILS', 'ROOT_CAUSE_CATEGORY', 'ROOT_CAUSE_TYPE', 'ROOT_CAUSE_TYPE_OTHER']

FAILURE:
['FAILURE_TYPE', 'FAILURE_SUBTYPE', 'EQ_FAILURE_TYPE', 'POWER_FAILURE_IND', 'ESD_SYSTEM_FAILURE_IND', 'FAILURE_DETAILS', 'RELATED_FAILURE_FOLLOW_IND']

INJUR:
['INJURY_IND', 'NUM_EMP_INJURIES', 'NUM_CONTR_INJURIES', 'NUM_ER_INJURIES', 'NUM_WORKER_INJURIES', 'NUM_GP_INJURIES', 'INJURE', 'NUM_INJURED_TREATED_BY_EMT']

FATAL:
['FATALITY_IND', 'NUM_EMP_FATALITIES', 'NUM_CONTR_FATALITIES', 'NUM_ER_FATALITIES', 'NUM_WORKER_FATALITIES', 'NUM_GP_FATALITIES', 'FATAL']

RELEASE:
['COMMODITY_RELEASED_TYPE', 'UNINTENTIONAL_RELEASE_BBLS', 'INTENTIONAL_RELEASE_BBLS', 'RELEASE_TYPE', 'RELEASE_TYPE_DETAILS', 'AMOUNT_RELEASED', 'EST_COST_GAS_RELEASED']

ACCIDENT_TYPE:
['OFF_AREA_ACCIDENT_TYPE']

REPORT_TYPE:
['REPORT_TYPE']

DATE:
['REPORT_RECEIVED_DATE', 'LOCAL_DATETIME', 'INCIDENT_IDENTIFIED_DATETIME', 'SHUTDOWN_DATETIME', 'RESTART_DATETIME', 'INITIAL

In [10]:
print(df["ACCIDENT_DETAILS"].head(10).to_string(index=False))

                             NaN
                             NaN
                             NaN
                             NaN
                             NaN
                             NaN
                             NaN
TRUCK DRIVER AT THE LOADING RACK
                             NaN
                             NaN


In [11]:
# Check missing values in important columns

text_columns = [
    "ACCIDENT_DETAILS",
    "FAILURE_DETAILS",
    "CAUSE",
    "CAUSE_DETAILS",
    "ROOT_CAUSE_CATEGORY",
    "ROOT_CAUSE_TYPE",
    "ROOT_CAUSE_TYPE_OTHER"
]

for col in text_columns:
    if col in df.columns:
        print(f"\n{col}")
        print("Available records:", df[col].notna().sum())
        print("Missing records:", df[col].isna().sum())
        print("Sample values:")
        print(df[col].dropna().head(3).to_string(index=False))


ACCIDENT_DETAILS
Available records: 278
Missing records: 5731
Sample values:
TRUCK DRIVER AT THE LOADING RACK
                     THIRD PARTY
      UNDETERMINED AT THIS TIME.

FAILURE_DETAILS
Available records: 330
Missing records: 5679
Sample values:
           STILL UNDER INVESTIGATION
SEAL FLUSH FILTER POT O-RING FAILURE
  RELIEF VALVE FAILED DUE TO EROSION

CAUSE
Available records: 6009
Missing records: 0
Sample values:
CORROSION FAILURE
EQUIPMENT FAILURE
EQUIPMENT FAILURE

CAUSE_DETAILS
Available records: 6009
Missing records: 0
Sample values:
            EXTERNAL CORROSION
       OTHER EQUIPMENT FAILURE
PUMP OR PUMP-RELATED EQUIPMENT

ROOT_CAUSE_CATEGORY
Available records: 198
Missing records: 5811
Sample values:
NOTIFICATION ISSUE
NOTIFICATION ISSUE
NOTIFICATION ISSUE

ROOT_CAUSE_TYPE
Available records: 199
Missing records: 5810
Sample values:
NO NOTIFICATION MADE TO THE ONE-CALL CENTER/811
NO NOTIFICATION MADE TO THE ONE-CALL CENTER/811
 EXCAVATOR DUG OUTSIDE AREA DESCRIBED O

In [12]:
# 1. Count distinct cause categories
print("Unique CAUSE values:", df["CAUSE"].nunique())
print("Unique CAUSE_DETAILS values:", df["CAUSE_DETAILS"].nunique())

# 2. Most frequent broad causes
print("\nTop 15 causes:")
print(df["CAUSE"].value_counts().head(15))

# 3. Most frequent detailed causes
print("\nTop 15 detailed causes:")
print(df["CAUSE_DETAILS"].value_counts().head(15))

# 4. Check how many descriptions are available
print("\nText availability:")
print("Accident descriptions:", df["ACCIDENT_DETAILS"].notna().sum())
print("Failure descriptions:", df["FAILURE_DETAILS"].notna().sum())

# 5. Check whether root-cause categories are balanced
print("\nRoot cause categories:")
print(df["ROOT_CAUSE_CATEGORY"].value_counts().head(15))

Unique CAUSE values: 8
Unique CAUSE_DETAILS values: 41

Top 15 causes:
CAUSE
EQUIPMENT FAILURE                   2740
CORROSION FAILURE                   1318
INCORRECT OPERATION                  832
MATERIAL FAILURE OF PIPE OR WELD     405
NATURAL FORCE DAMAGE                 263
EXCAVATION DAMAGE                    199
OTHER ACCIDENT CAUSE                 127
OTHER OUTSIDE FORCE DAMAGE           125
Name: count, dtype: int64

Top 15 detailed causes:
CAUSE_DETAILS
INTERNAL CORROSION                                                                                                                  845
NON-THREADED CONNECTION FAILURE                                                                                                     729
PUMP OR PUMP-RELATED EQUIPMENT                                                                                                      594
EXTERNAL CORROSION                                                                                                        

In [13]:
# 1. Count records with accident and failure descriptions
accident_text = df["ACCIDENT_DETAILS"].fillna("").str.strip()
failure_text = df["FAILURE_DETAILS"].fillna("").str.strip()

print("Accident descriptions:", accident_text.ne("").sum())
print("Failure descriptions:", failure_text.ne("").sum())

# 2. Combine the available descriptions
df["INCIDENT_TEXT"] = (
    accident_text + " " + failure_text
).str.strip()

# 3. Count records with usable text
text_df = df[
    df["INCIDENT_TEXT"].ne("") &
    df["CAUSE"].notna()
].copy()

print("\nRecords with text and cause labels:", len(text_df))
print("Unique cause labels:", text_df["CAUSE"].nunique())

# 4. Check cause distribution in the usable records
print("\nCause distribution:")
print(text_df["CAUSE"].value_counts())

# 5. Preview the usable records
display(text_df[["INCIDENT_TEXT", "CAUSE"]].head(10))

Accident descriptions: 278
Failure descriptions: 330

Records with text and cause labels: 591
Unique cause labels: 8

Cause distribution:
CAUSE
EQUIPMENT FAILURE                   447
CORROSION FAILURE                    69
INCORRECT OPERATION                  24
MATERIAL FAILURE OF PIPE OR WELD     15
OTHER OUTSIDE FORCE DAMAGE           13
OTHER ACCIDENT CAUSE                  8
NATURAL FORCE DAMAGE                  8
EXCAVATION DAMAGE                     7
Name: count, dtype: int64


C:\Users\mohammad rayyan\AppData\Local\Temp\ipykernel_18632\2654153800.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["INCIDENT_TEXT"] = (


,INCIDENT_TEXT,CAUSE
1,STILL UNDER INVESTIGATION,EQUIPMENT FAILURE
7,TRUCK DRIVER AT THE LOADING RACK,EQUIPMENT FAILURE
8,SEAL FLUSH FILTER POT O-RING FAILURE,EQUIPMENT FAILURE
23,THIRD PARTY,OTHER ACCIDENT CAUSE
28,RELIEF VALVE FAILED DUE TO EROSION,EQUIPMENT FAILURE
30,INVESTIGATION ONGOING.,EQUIPMENT FAILURE
71,LEAK AT THE TANK MIXER SEAL,EQUIPMENT FAILURE
90,UNDETERMINED AT THIS TIME.,OTHER ACCIDENT CAUSE
91,CORROSION COUPON HOLDER,EQUIPMENT FAILURE
105,THIRD PARTY,EQUIPMENT FAILURE


In [14]:
# Work with a fresh, smaller DataFrame
text_df = text_df.copy()

# Remove empty text and normalize whitespace
text_df["INCIDENT_TEXT"] = (
    text_df["INCIDENT_TEXT"]
    .fillna("")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Check text length
text_df["TEXT_LENGTH"] = text_df["INCIDENT_TEXT"].str.len()

print("Total records:", len(text_df))
print("Average text length:", round(text_df["TEXT_LENGTH"].mean(), 2))
print("Median text length:", text_df["TEXT_LENGTH"].median())

print("\nShortest text records:")
display(
    text_df.sort_values("TEXT_LENGTH")[
        ["INCIDENT_TEXT", "CAUSE", "TEXT_LENGTH"]
    ].head(15)
)

print("\nMost frequent incident texts:")
print(text_df["INCIDENT_TEXT"].value_counts().head(20))

print("\nText length distribution:")
print(text_df["TEXT_LENGTH"].describe())

Total records: 591
Average text length: 48.04
Median text length: 36.0

Shortest text records:


,INCIDENT_TEXT,CAUSE,TEXT_LENGTH
2382,TANK,EQUIPMENT FAILURE,4
5631,PREQB,OTHER OUTSIDE FORCE DAMAGE,5
4059,CLAMP,EQUIPMENT FAILURE,5
3877,NIPPLE,EQUIPMENT FAILURE,6
3905,NIPPLE,EQUIPMENT FAILURE,6
2213,O-RING,EQUIPMENT FAILURE,6
2314,HEATER,EQUIPMENT FAILURE,6
4027,FLANGE,EQUIPMENT FAILURE,6
735,SAMPLER,EQUIPMENT FAILURE,7
4191,FLEX HOSE,EQUIPMENT FAILURE,9



Most frequent incident texts:
INCIDENT_TEXT
THIRD PARTY OPERATOR             11
MIXER SEAL                        7
THIRD PARTY                       6
TANK MIXER SEAL                   6
STEM SEAL                         5
3RD PARTY OPERATOR                3
PRESSURE GAUGE                    3
FLEX HOSE                         3
ANOTHER OPERATOR                  3
NOTIFICATION FROM THIRD PARTY     3
INJECTION PLANT OPERATOR          3
REFINERY OPERATIONS PERSONNEL     3
CHECK VALVE                       3
THIRD-PARTY OPERATOR              2
MREGZ SENSOR                      2
LANDOWNER REPORTED                2
GAS DETECTOR ALARM                2
VALVE PACKING LEAK                2
THIRD PARTY AERIAL PATROL         2
THIRD PARTY TRUCK DRIVER          2
Name: count, dtype: int64

Text length distribution:
count    591.000000
mean      48.035533
std       50.393540
min        4.000000
25%       22.000000
50%       36.000000
75%       57.000000
max      422.000000
Name: TEXT_LENGTH, dty

In [15]:
from IPython.display import display

# Show the dataset dimensions
print("Total rows and columns:", df.shape)

# Display the first 20 actual records
display(df.head(20))

Total rows and columns: (6009, 649)


,REPORT_RECEIVED_DATE,IYEAR,REPORT_NUMBER,SUPPLEMENTAL_NUMBER,REPORT_TYPE,OPERATOR_ID,NAME,OPERATOR_STREET_ADDRESS,OPERATOR_CITY_NAME,OPERATOR_STATE_ABBREVIATION,...,PREPARED_DATE,LOCAL_CONTACT_NAME,LOCAL_CONTACT_EMAIL,LOCAL_CONTACT_TELEPHONE,AUTHORIZER_NAME,AUTHORIZER_TELEPHONE,AUTHORIZER_TITLE,AUTHORIZER_EMAIL,NARRATIVE,INCIDENT_TEXT
0,8/27/2026,2026,20260196,42679,ORIGINAL,30020,W & T OFFSHORE INC,5718 WESTHEIMER ROAD SUITE 700,HOUSTON,TX,...,8/27/2026,KIRK BORDELON,kbordelon@wtoffshore.com,832-486-3697,JOHN POOLE,713-624-7338,VICE PRESIDENT HSER,jpoole@wtoffshore.com,"DURING ROUTINE INSPECTION OF THE PLATFORM, PLA...",
1,8/30/2026,2026,20260200,42687,ORIGINAL,32109,"ONEOK NGL PIPELINE, LLC",100 WEST FIFTH STREET,TULSA,OK,...,8/28/2026,NaN,NaN,NaN,RYAN MCCOY,5807410185,NaN,ryan.mccoy@oneok.com,"ON 8/1/26, ONEOK PCC WAS NOTIFIED BY A LANDOWN...",STILL UNDER INVESTIGATION
2,8/28/2026,2026,20260199,42684,ORIGINAL,31957,KINDER MORGAN WINK PIPELINE LLC,"1001 LOUISIANA ST, SUITE 1000",HOUSTON,TX,...,8/28/2026,NaN,NaN,NaN,DAVID TOWNSEND,7134204491,DIRECTOR - OPERATIONS,david_townsend@kindermorgan.com,"ON JULY 30, 2026, AT 19:10CST, A KINDER MORGAN...",
3,8/13/2026,2026,20260188,42624,ORIGINAL FINAL,39401,GREAT SALT PLAINS PIPELINE LLC,"8301 E 21ST ST, SUITE 370",WICHITA,KS,...,8/13/2026,JOE VAVERKA,jvaverka@mvpurchasing.com,405-508-0410,ADAM COWART,6208860082,MANAGER OF EHS AND COMPLIANCE,acowart@mvpurchasing.com,"ON JULY 29, 2026, THE CONTROL ROOM RECEIVED A ...",
4,8/27/2026,2026,20260197,42682,ORIGINAL,39377,"WHITNEY OIL & GAS, LLC",1415 LOUSIANA ST,HOUSTON,TX,...,8/27/2026,TAYLOR WALKER,twalker@torrentoil.com,601-498-7289,RUSSELL STEINER,3375190843,REGULATORY AND COMPLIANCE MANAGER,rsteiner@torrentoil.com,"ON JULY 29, 2026, OPERATING PERSONNEL NOTICED ...",
5,8/28/2026,2026,20260198,42683,ORIGINAL,395,AMOCO OIL CO,30 S. WACKER DRIVE,CHICAGO,IL,...,8/28/2026,NaN,NaN,NaN,JIM BRUEN,872-245-2317,DOT COMPLIANCE TEAM LEAD,jim.bruen@bp.com,THE TULSA CONTROL CENTER RECEIVED A COMPLAINT ...,
6,8/18/2026,2026,20260191,42652,ORIGINAL,473,"WESTERN MIDSTREAM PARTNERS, LP",9950 WOODLOCH FOREST DR SUITE 2800,THE WOODLANDS,TX,...,8/18/2026,NaN,NaN,NaN,WILLIAM NELSON,9792854071,NaN,william.nelson@westernmidstream.com,APPROXIMATELY 12 BARRELS OF CRUDE OIL WAS LOST...,
7,8/25/2026,2026,20260195,42670,ORIGINAL FINAL,22610,"MAGELLAN PIPELINE COMPANY, LP",100 WEST 5TH STREET,TULSA,OK,...,8/25/2026,NaN,NaN,NaN,RYAN MCCOY,918-381-8066,NaN,gabe.chronister@oneok.com,"ON 7/26, A TRUCK DRIVER REPORTED A RELEASE AT ...",TRUCK DRIVER AT THE LOADING RACK
8,8/24/2026,2026,20260194,42669,ORIGINAL,31618,ENTERPRISE PRODUCTS OPERATING LLC,1100 LOUISIANA STREET,HOUSTON,TX,...,8/24/2026,NAVJOT KAUR,nkaur@eprod.com,7133814526,SUZIE DAVIS,7133815585,SENIOR MANAGER COMPLIANCE,smdavis@eprod.com,"ON 7/24/2026 AT 08:34 AM CST, ENTERPRISE TECHN...",SEAL FLUSH FILTER POT O-RING FAILURE
9,8/19/2026,2026,20260192,42655,ORIGINAL,39823,TARGA DOWNSTREAM LLC,811 LOUISIANA ST. SUITE 2100,HOUSTON,TX,...,8/19/2026,JORDAN NORTH,jnorth@targaresources.com,9405779500,JORDAN NORTH,9405779500,SPECIALIST ESH,jnorth@targaresources.com,ON 7/24/26 AT 2 AM TARGA NGL CONTROL ROOM RECE...,


ModuleNotFoundError: No module named 'pathlab'